# Authorization and Policy Enforcement

Build a deterministic policy decision point (PDP) and enforcement point (PEP) for support tools. The model proposes; trusted code supplies identity and current resource facts, decides, enforces obligations, controls effects, and records evidence.

## Scenario and safety boundary

The notebook uses Pydantic, the official MCP SDK, and in-memory fixtures. It proves local policy invariants but not distributed durability, external-engine correctness, TLS, database transactions, or production message delivery.

In [ ]:
import runpy
from datetime import UTC, datetime
from pathlib import Path

module = runpy.run_path(Path('lab.py'))
NOW = datetime(2026, 9, 27, 16, 0, tzinfo=UTC)
enforcement, agent, supervisor, session = module['default_environment'](NOW)
agent

## PARC and trusted inputs

A decision evaluates principal, action, resource, and context. Subject, tenant, roles, permissions, purpose, device posture, risk, current time, resource ownership/state, and relationship revision come from trusted application/PIP state—not tool arguments.

In [ ]:
from mcp import Client
server = module['build_mcp_server'](enforcement, session)
async with Client(server) as client:
    listed = await client.list_tools()
schemas = {tool.name: tool.input_schema for tool in listed.tools}
assert all('tenant_id' not in str(schema) for schema in schemas.values())
schemas

## RBAC + ABAC + ReBAC

A support role is only a candidate grant. Permission narrows the action; tenant, purpose, lifecycle, device, and risk are attributes; assignment is a relationship. Default deny and guardrails apply before any allow.

In [ ]:
RequestContext = module['RequestContext']
Risk = module['Risk']

def context(request_id, purpose='support'):
    return RequestContext(
        request_id=request_id, purpose=purpose, managed_device=True,
        risk=Risk.LOW, now=NOW
    )

def decide(principal, action, ticket_id, request_id, purpose='support'):
    ticket = enforcement.tickets.get(ticket_id)
    request = enforcement._request(
        principal=principal, action=action, ticket=ticket,
        context=context(request_id, purpose),
    )
    return enforcement.pdp.decide(request)

matrix = {
    'assigned read': decide(agent, 'ticket.read', 'acme-100', 'matrix-1'),
    'cross tenant': decide(agent, 'ticket.read', 'globex-200', 'matrix-2'),
    'wrong purpose': decide(agent, 'ticket.read', 'acme-100', 'matrix-3', 'audit'),
    'unknown action': decide(agent, 'filesystem.delete', 'acme-100', 'matrix-4'),
}
{name: (value.effect.value, value.reason_code) for name, value in matrix.items()}

## An allow is incomplete until obligations are enforced

The PDP requires email redaction and internal-note omission. The PEP applies those obligations before returning data and attaches a decision ID for correlation.

In [ ]:
visible = enforcement.read_ticket(agent, 'acme-100', context('read-1'))
assert visible['customer_email'] == '[redacted]'
assert 'internal_note' not in visible
visible

## Approval is a bound, expiring, single-use capability

A reply proposal has a stable logical operation ID, current resource version, validated body, and canonical digest. A separate trusted supervisor issues a receipt bound to the exact subject, tenant, action, ticket, operation, digest, resource version, policy version, and expiry.

In [ ]:
proposal = enforcement.prepare_reply(
    agent, ticket_id='acme-100', operation_id='op-notebook-1',
    body='We confirmed the invoice retry completed.',
    context=context('prepare-1'),
)
receipt = enforcement.approvals.issue(
    approver=supervisor, principal=agent, proposal=proposal,
    policy_version=module['POLICY_VERSION'], now=NOW,
)
{
    'operation_id': proposal.operation_id,
    'proposal_digest': proposal.digest,
    'receipt_id': receipt.receipt_id,
    'approver': receipt.approver_subject,
    'state': receipt.state.value,
}

In [ ]:
first, first_deduplicated = enforcement.send_reply(
    agent, proposal=proposal, approval_id=receipt.receipt_id,
    context=context('send-1'),
)
retry, retry_deduplicated = enforcement.send_reply(
    agent, proposal=proposal, approval_id=receipt.receipt_id,
    context=context('send-retry'),
)
assert first == retry
assert first_deduplicated is False and retry_deduplicated is True
assert len(enforcement.executions) == 1
{
    'operation_id': first.operation_id,
    'status': first.status,
    'approval_state': receipt.state.value,
    'retry_deduplicated': retry_deduplicated,
}

## Failure injection: mutate the approved content

Changing the body while retaining the old digest fails strict proposal validation before authorization. Reusing the same operation ID with a newly valid digest would still conflict with the completed operation record.

In [ ]:
from pydantic import ValidationError
tampered = proposal.model_dump(mode='json') | {'body': 'Send credentials instead'}
try:
    module['ReplyProposal'].model_validate(tampered)
    tamper_result = 'unexpected allow'
except ValidationError as exc:
    tamper_result = 'denied: proposal digest mismatch'
tamper_result

## Evaluate the policy, not a vague accuracy score

False allows are counted over expected denials; false denies are counted over expected allows. Keep both numerators and denominators and slice them by action, tenant, role, resource class, policy version, and failure mode.

In [ ]:
PolicyCase = module['PolicyCase']
Effect = module['Effect']
no_permission = agent.model_copy(update={'permissions': frozenset()})
wrong_tenant = agent.model_copy(update={'tenant_id': 'globex'})
cases = (
    PolicyCase('assigned read', agent, 'ticket.read', 'acme-100', 'support', Effect.ALLOW),
    PolicyCase('draft', agent, 'ticket.reply.draft', 'acme-100', 'support', Effect.ALLOW),
    PolicyCase('no permission', no_permission, 'ticket.read', 'acme-100', 'support', Effect.DENY),
    PolicyCase('wrong tenant', wrong_tenant, 'ticket.read', 'acme-100', 'support', Effect.DENY),
    PolicyCase('wrong purpose', agent, 'ticket.read', 'acme-100', 'audit', Effect.DENY),
)
report = module['evaluate_cases'](enforcement, cases, NOW)
assert report.false_allows == 0 and report.false_denies == 0
report

## Normal → attack → defense → retest

Run the canonical scenario through real MCP tool calls. It proves cross-tenant and unapproved denials, obligation enforcement, exact approval, single execution, idempotent retry, and redacted audit evidence.

In [ ]:
evidence = await module['run_scenario']()
assert evidence['cross_tenant_denied']
assert evidence['unapproved_send_denied']
assert evidence['execution_count'] == 1
assert evidence['reply_body_in_audit'] is False
evidence

## Production upgrades and exercises

1. Implement an OPA/Rego or Cedar adapter behind the same PDP contract and preserve fail-closed PEP behavior.
2. Move assignments into OpenFGA or SpiceDB; propagate model and consistency revisions.
3. Add policy bundle/schema validation, shadow evaluation, canary activation, and rollback.
4. Add list/search authorization without exposing denied records before filtering.
5. Replace the execution fixture with a durable idempotency ledger and reconcile timeout-after-commit.
6. Run `python -m pytest -q tests/test_course_07_authorization_policy.py`.

## Reflection

Which facts in your own MCP tools still come from caller-controlled arguments? Which protected paths can bypass the PEP? What policy/relationship consistency is required immediately after revocation, and how would an exact approved operation recover after an unknown downstream outcome?